# Trip Events Analysis

## Objective

This notebook analyzes trip event data to evaluate:

- Event lifecycle
- Data quality
- Duplicate records
- Missing values
- Event ordering
- Timestamp consistency
- Business rule validation

The findings from this analysis will be used to design reliable ETL pipelines and data quality checks.

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [3]:
DATA_PATH = Path("../data/raw/raw_trips_events.csv")

events = pd.read_csv(DATA_PATH)

events.head()

,event_id,trip_id,rider_id,driver_id,city_id,status,fare_amount,surge_multiplier,distance_km,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,vehicle_type,promo_code_id,event_ts,ingested_at,payment_method
0,700106,5102,90102,3007.0,34,requested,0.0,1.5,NaN,41.010000,29.000000,NaN,NaN,premium,NaN,2026-05-08 23:40:00,2026-05-08 23:42:00,NaN
1,700107,5102,90102,3007.0,34,driver_assigned,0.0,1.5,NaN,41.010000,29.000000,NaN,NaN,premium,NaN,2026-05-08 23:41:00,2026-05-08 23:43:00,NaN
2,700108,5102,90102,3007.0,34,picked_up,0.0,1.5,NaN,41.010000,29.000000,NaN,NaN,premium,NaN,2026-05-08 23:44:00,2026-05-08 23:46:00,NaN
3,700001,5001,90001,NaN,34,requested,0.0,1.0,NaN,41.028272,29.078436,NaN,NaN,comfort,NaN,2026-05-09 06:47:00,2026-05-09 06:49:00,NaN
4,700002,5001,90001,3001.0,34,driver_assigned,0.0,1.0,NaN,41.028272,29.078436,NaN,NaN,comfort,NaN,2026-05-09 06:47:23,2026-05-09 06:49:23,NaN


In [4]:
events.shape

(142, 18)

In [5]:
events.columns.tolist()

['event_id',
 'trip_id',
 'rider_id',
 'driver_id',
 'city_id',
 'status',
 'fare_amount',
 'surge_multiplier',
 'distance_km',
 'pickup_lat',
 'pickup_lng',
 'dropoff_lat',
 'dropoff_lng',
 'vehicle_type',
 'promo_code_id',
 'event_ts',
 'ingested_at',
 'payment_method']

In [6]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 142 entries, 0 to 141
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   event_id          142 non-null    int64  
 1   trip_id           142 non-null    int64  
 2   rider_id          142 non-null    int64  
 3   driver_id         110 non-null    float64
 4   city_id           142 non-null    int64  
 5   status            142 non-null    object 
 6   fare_amount       141 non-null    float64
 7   surge_multiplier  142 non-null    float64
 8   distance_km       35 non-null     float64
 9   pickup_lat        142 non-null    float64
 10  pickup_lng        142 non-null    float64
 11  dropoff_lat       35 non-null     float64
 12  dropoff_lng       35 non-null     float64
 13  vehicle_type      142 non-null    object 
 14  promo_code_id     0 non-null      float64
 15  event_ts          142 non-null    object 
 16  ingested_at       142 non-null    object 
 1

In [8]:
events.describe(include="all")

,event_id,trip_id,rider_id,driver_id,city_id,status,fare_amount,surge_multiplier,distance_km,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,vehicle_type,promo_code_id,event_ts,ingested_at,payment_method
count,142.000000,142.000000,142.000000,110.000000,142.000000,142,141.000000,142.000000,35.000000,142.000000,142.000000,35.000000,35.000000,142,0.0,142,142,21
unique,NaN,NaN,NaN,NaN,NaN,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,140,140,3
top,NaN,NaN,NaN,NaN,NaN,requested,NaN,NaN,NaN,NaN,NaN,NaN,NaN,comfort,NaN,2026-05-09 09:30:46,2026-05-09 09:32:46,card
freq,NaN,NaN,NaN,NaN,NaN,36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53,NaN,2,2,10
mean,700071.500000,5040.387324,90040.387324,3005.372727,30.556338,NaN,39.587376,1.170423,14.152000,39.569036,28.963418,40.783560,29.778336,NaN,NaN,NaN,NaN,NaN
std,41.135953,42.877171,42.877171,3.410343,9.394071,NaN,82.251016,0.218242,7.758163,6.784186,5.195824,0.453200,1.537291,NaN,NaN,NaN,NaN,NaN
min,700001.000000,5001.000000,90001.000000,3000.000000,6.000000,NaN,-45.000000,1.000000,2.550000,0.000000,0.000000,39.855717,28.911188,NaN,NaN,NaN,NaN,NaN
25%,700036.250000,5009.250000,90009.250000,3002.000000,34.000000,NaN,0.000000,1.000000,7.955000,39.947189,29.010000,40.928618,29.001644,NaN,NaN,NaN,NaN,NaN
50%,700071.500000,5018.000000,90018.000000,3005.000000,34.000000,NaN,0.000000,1.000000,12.400000,41.000000,29.037974,40.993416,29.054224,NaN,NaN,NaN,NaN,NaN
75%,700106.750000,5102.000000,90102.000000,3008.000000,34.000000,NaN,0.000000,1.500000,21.430000,41.029568,29.095597,41.046036,29.084893,NaN,NaN,NaN,NaN,NaN


In [9]:
missing = (
    events.isnull()
    .sum()
    .to_frame("Missing Values")
)

missing["Percentage"] = (
    missing["Missing Values"] / len(events) * 100
).round(2)

missing.sort_values(
    by="Missing Values",
    ascending=False
)

,Missing Values,Percentage
promo_code_id,142,100.00
payment_method,121,85.21
dropoff_lat,107,75.35
distance_km,107,75.35
dropoff_lng,107,75.35
driver_id,32,22.54
fare_amount,1,0.70
ingested_at,0,0.00
event_ts,0,0.00
vehicle_type,0,0.00


In [10]:
events["status"].value_counts()

requested              36
picked_up              35
completed              35
driver_assigned        34
no_driver_found         1
cancelled_by_driver     1
Name: status, dtype: int64

In [11]:
events["trip_id"].nunique()

36

In [12]:
events.groupby("trip_id").size().describe()

count    36.000000
mean      3.944444
std       0.474760
min       2.000000
25%       4.000000
50%       4.000000
75%       4.000000
max       5.000000
dtype: float64

In [13]:
events["event_id"].duplicated().sum()

0

In [14]:
events.duplicated().sum()

0

In [16]:
events["event_ts"] = pd.to_datetime(events["event_ts"])
events["ingested_at"] = pd.to_datetime(events["ingested_at"])

In [17]:
(events["ingested_at"] - events["event_ts"]).describe()

count                          142
mean     0 days 00:00:23.028169014
std      0 days 00:50:42.198950282
min              -1 days +21:02:10
25%                0 days 00:02:00
50%                0 days 00:02:00
75%                0 days 00:02:00
max                0 days 08:07:00
dtype: object

In [18]:
events[
    events["fare_amount"] < 0
]

,event_id,trip_id,rider_id,driver_id,city_id,status,fare_amount,surge_multiplier,distance_km,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,vehicle_type,promo_code_id,event_ts,ingested_at,payment_method
54,700142,5111,90111,3003.0,34,completed,-45.0,1.0,10.2,41.02,29.02,41.05,29.06,comfort,NaN,2026-05-09 12:55:39,2026-05-09 12:57:39,card


In [19]:
events[
    (events["pickup_lat"] == 0)
    | (events["pickup_lng"] == 0)
]

,event_id,trip_id,rider_id,driver_id,city_id,status,fare_amount,surge_multiplier,distance_km,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,vehicle_type,promo_code_id,event_ts,ingested_at,payment_method
83,700126,5107,90107,3001.0,34,requested,0.0,1.2,NaN,0.0,0.0,NaN,NaN,comfort,NaN,2026-05-09 17:10:00,2026-05-09 17:12:00,NaN
84,700127,5107,90107,3001.0,34,driver_assigned,0.0,1.2,NaN,0.0,0.0,NaN,NaN,comfort,NaN,2026-05-09 17:10:45,2026-05-09 17:12:45,NaN
85,700128,5107,90107,3001.0,34,picked_up,0.0,1.2,NaN,0.0,0.0,NaN,NaN,comfort,NaN,2026-05-09 17:13:45,2026-05-09 17:15:45,NaN
86,700129,5107,90107,3001.0,34,completed,98.0,1.2,7.3,0.0,0.0,41.05,29.06,comfort,NaN,2026-05-09 17:23:45,2026-05-09 17:25:45,card


In [20]:
events = events.sort_values(["trip_id", "event_ts"])

events.groupby("trip_id")["status"].apply(list)

trip_id
5001    [requested, driver_assigned, picked_up, comple...
5002    [requested, driver_assigned, picked_up, comple...
5003    [requested, driver_assigned, picked_up, comple...
5004    [requested, driver_assigned, picked_up, comple...
5005    [requested, driver_assigned, picked_up, comple...
5006    [requested, driver_assigned, picked_up, comple...
5007    [requested, driver_assigned, picked_up, comple...
5008    [requested, driver_assigned, picked_up, comple...
5009    [requested, driver_assigned, picked_up, comple...
5010    [requested, driver_assigned, picked_up, comple...
5011    [requested, driver_assigned, picked_up, comple...
5012    [requested, driver_assigned, picked_up, comple...
5013    [requested, driver_assigned, picked_up, comple...
5014    [requested, driver_assigned, picked_up, comple...
5015    [requested, driver_assigned, picked_up, comple...
5016    [requested, driver_assigned, picked_up, comple...
5017    [requested, driver_assigned, picked_up, comple...
5018  

In [21]:
events[
    events["ingested_at"] < events["event_ts"]
]

,event_id,trip_id,rider_id,driver_id,city_id,status,fare_amount,surge_multiplier,distance_km,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,vehicle_type,promo_code_id,event_ts,ingested_at,payment_method
90,700130,5108,90108,3011.0,34,requested,0.0,1.0,NaN,41.02,29.01,NaN,NaN,economy,NaN,2026-05-09 21:05:00,2026-05-09 18:08:00,NaN
91,700131,5108,90108,3011.0,34,driver_assigned,0.0,1.0,NaN,41.02,29.01,NaN,NaN,economy,NaN,2026-05-09 21:05:50,2026-05-09 18:09:00,NaN
92,700132,5108,90108,3011.0,34,picked_up,0.0,1.0,NaN,41.02,29.01,NaN,NaN,economy,NaN,2026-05-09 21:09:50,2026-05-09 18:12:00,NaN
94,700133,5108,90108,3011.0,34,completed,120.0,1.0,9.6,41.02,29.01,41.06,29.05,economy,NaN,2026-05-09 21:24:50,2026-05-09 18:30:00,wallet


In [22]:
events.groupby("status")["payment_method"].count()

status
cancelled_by_driver     0
completed              21
driver_assigned         0
no_driver_found         0
picked_up               0
requested               0
Name: payment_method, dtype: int64

In [23]:
events.groupby("status")["distance_km"].count()

status
cancelled_by_driver     0
completed              35
driver_assigned         0
no_driver_found         0
picked_up               0
requested               0
Name: distance_km, dtype: int64

In [24]:
events.groupby("status")["driver_id"].count()

status
cancelled_by_driver     1
completed              34
driver_assigned        34
no_driver_found         0
picked_up              34
requested               7
Name: driver_id, dtype: int64

In [25]:
events["event_ts"] = pd.to_datetime(events["event_ts"])
events["ingested_at"] = pd.to_datetime(events["ingested_at"])

In [ ]:
#Business-level duplicate event

business_event_columns = [
    "trip_id",
    "status",
    "event_ts"
]

events[
    events.duplicated(
        subset=business_event_columns,
        keep=False
    )
].sort_values(["trip_id", "event_ts"])

,event_id,trip_id,rider_id,driver_id,city_id,status,fare_amount,surge_multiplier,distance_km,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,vehicle_type,promo_code_id,event_ts,ingested_at,payment_method
28,700104,5101,90101,3005.0,34,completed,178.5,1.2,12.4,41.063837,29.059224,41.057712,29.079785,comfort,NaN,2026-05-09 09:30:46,2026-05-09 09:32:46,card
29,700105,5101,90101,3005.0,34,completed,178.5,1.2,12.4,41.020000,29.010000,41.050000,29.060000,comfort,NaN,2026-05-09 09:30:46,2026-05-09 09:32:46,card


In [27]:
#Completed trip with NULL fare
events[
    (events["status"] == "completed")
    & (events["fare_amount"].isna())
]

,event_id,trip_id,rider_id,driver_id,city_id,status,fare_amount,surge_multiplier,distance_km,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,vehicle_type,promo_code_id,event_ts,ingested_at,payment_method
42,700113,5103,90103,3002.0,34,completed,NaN,1.0,8.9,41.03,29.02,41.06,29.05,economy,NaN,2026-05-09 11:20:40,2026-05-09 11:22:40,card


In [28]:
#Out-of-range fare
events[
    (events["fare_amount"] < 0)
    | (events["fare_amount"] > 300)
]

,event_id,trip_id,rider_id,driver_id,city_id,status,fare_amount,surge_multiplier,distance_km,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,vehicle_type,promo_code_id,event_ts,ingested_at,payment_method
117,700024,5006,90006,3006.0,34,completed,317.12,1.5,18.64,40.955138,28.964857,41.042930,28.911188,comfort,NaN,2026-05-09 20:59:24,2026-05-09 21:01:24,NaN
49,700040,5010,90010,3010.0,6,completed,307.30,1.0,24.77,39.947189,32.826196,39.900768,32.760641,economy,NaN,2026-05-09 11:53:58,2026-05-09 11:55:58,NaN
66,700048,5012,90012,3000.0,34,completed,305.41,1.2,4.23,41.050564,29.095597,41.005233,28.925152,premium,NaN,2026-05-09 14:19:20,2026-05-09 14:21:20,NaN
54,700142,5111,90111,3003.0,34,completed,-45.00,1.0,10.20,41.020000,29.020000,41.050000,29.060000,comfort,NaN,2026-05-09 12:55:39,2026-05-09 12:57:39,card


In [29]:
#Completed trip with NULL driver
events[
    (events["status"] == "completed")
    & (events["driver_id"].isna())
]

,event_id,trip_id,rider_id,driver_id,city_id,status,fare_amount,surge_multiplier,distance_km,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,vehicle_type,promo_code_id,event_ts,ingested_at,payment_method
79,700125,5106,90106,NaN,34,completed,66.0,1.0,5.2,41.01,29.02,41.03,29.04,economy,NaN,2026-05-09 16:02:00,2026-05-09 16:04:00,cash


In [30]:
#Status regression
status_rank = {
    "requested": 1,
    "driver_assigned": 2,
    "picked_up": 3,
    "completed": 4
}

events["status_rank"] = events["status"].map(status_rank)

ordered_events = events.sort_values(
    ["trip_id", "event_ts", "event_id"]
).copy()

ordered_events["previous_status_rank"] = (
    ordered_events
    .groupby("trip_id")["status_rank"]
    .shift(1)
)

ordered_events[
    ordered_events["status_rank"]
    < ordered_events["previous_status_rank"]
]

,event_id,trip_id,rider_id,driver_id,city_id,status,fare_amount,surge_multiplier,distance_km,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,vehicle_type,promo_code_id,event_ts,ingested_at,payment_method,status_rank,previous_status_rank
59,700118,5104,90104,3009.0,34,picked_up,0.0,1.0,NaN,41.0,29.03,NaN,NaN,comfort,NaN,2026-05-09 13:43:50,2026-05-09 13:45:50,NaN,3.0,4.0


In [31]:
#Trips spanning midnight
trip_dates = (
    events.groupby("trip_id")
    .agg(
        first_event_ts=("event_ts", "min"),
        last_event_ts=("event_ts", "max")
    )
)

trip_dates[
    trip_dates["first_event_ts"].dt.date
    != trip_dates["last_event_ts"].dt.date
]

,first_event_ts,last_event_ts
trip_id,,
5105,2026-05-09 23:52:00,2026-05-10 00:14:00


# Findings

## Data Quality Assessment

### Validated Constraints

- No fully duplicated rows were detected.
- `event_id` is unique at the CDC event level.
- `distance_km` is populated only for completed trip events.
- Non-null `payment_method` values appear only on completed trip events.

### Schema Evolution Observation

- `payment_method` is missing for a subset of completed trip events.
- The documented source schema states that this column was not present in older data.
- Historical NULL values in `payment_method` should therefore not automatically be classified as data quality failures.
- However, the appearance of a new source column should be detected and governed through schema-drift and data-contract controls.

### Data Quality Anomalies

- A completed status event for trip `5101` was delivered more than once with different `event_id` values, demonstrating a business-level duplicate that cannot be detected using `event_id` uniqueness alone.
- Trip `5103` contains a completed event with a NULL `fare_amount`.
- Fare values outside the assumed acceptable analytical range were detected, including values above 300 and one negative fare amount of -45.
- Trip `5106` completes with a NULL `driver_id`, representing an integrity issue and an early-arriving fact / late-arriving dimension scenario.
- Trip `5104` contains a status regression from `completed` back to `picked_up`. A latest-timestamp-only final-state rule would therefore produce an incorrect result.
- Trip `5105` spans midnight, starting on 2026-05-09 and ending on 2026-05-10.
- One trip contains invalid pickup coordinates `(0, 0)`, affecting multiple CDC events for the same trip.
- Some events have `ingested_at` earlier than `event_ts`. The multi-hour offset pattern should be investigated as a possible timezone inconsistency rather than silently corrected.

## Engineering Implications

- CDC deduplication must use a business-event key in addition to `event_id`.
- Final trip state must be derived using lifecycle-aware status precedence and must not rely only on the latest `event_ts`.
- Late and out-of-order events require a watermark and reprocessing window.
- Trips spanning midnight require an explicit attribution rule. The recommended approach is to attribute the trip to its `requested` date while allowing later events to update the same trip state.
- Completed trips with NULL fares must be quarantined because they violate the expected source contract.
- Facts with missing or unresolved drivers should not be dropped. An inferred or unknown dimension member should preserve referential integrity.
- Timestamp inconsistencies should trigger investigation and quarantine or alerting until the upstream timezone semantics are confirmed.

## Recommendations

- Deduplicate repeated business events using stable business attributes rather than relying only on `event_id`.
- Validate completed trips with `fare_amount IS NOT NULL` and `fare_amount >= 0`.
- Apply a documented fare-range distribution check and investigate extreme values.
- Validate geographic coordinate ranges and reject `(0, 0)` pickup coordinates.
- Detect lifecycle regressions using status precedence rules.
- Use a late-data watermark and reprocess affected trip IDs across the watermark window.
- Handle early-arriving facts through an inferred or unknown driver dimension member.
- Treat `payment_method` as a schema-evolved nullable attribute while enforcing schema-change notification through a data contract.
- Preserve invalid source events in Bronze and route offending records to quarantine rather than deleting them.